In [1]:
# ============================================================
# mBERT EN↔FR: Word-level Attention vs Optimal Transport (Euclidean)
# - Robust token→word aggregation (no index errors)
# - Per-layer word-level attention (FR rows × EN cols), blue circles on gold pairs
# - Per-layer word embeddings + OT (Euclidean or normalized Euclidean); toggle EMD vs Sinkhorn
# - CSVs + heatmaps + correlation curve
# - Plots HIDE placeholder labels like en_31 / fr_31 / f_31
# ============================================================

# (Optional) Limit threads on constrained machines
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import csv
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import torch
from transformers import BertTokenizerFast, BertModel
import ot  # pip install pot

# ----------------------------
# 1) Model & sentences
# ----------------------------
model_name = "bert-base-multilingual-cased"  # 12 layers
tokenizer = BertTokenizerFast.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True, output_hidden_states=True)
model.eval()

# Sentences (contain target gold pairs)
en_sentence = "The quick brown fox jumps over the lazy dog."
fr_sentence = "Le renard brun rapide saute par-dessus le chien paresseux."

# ----------------------------
# 2) Gold word pairs (EN, FR)
# ----------------------------
gold_pairs = [
    ("the", "le"),          # first "the" ↔ first "le"
    ("quick", "rapide"),
    ("brown", "brun"),
    ("fox", "renard"),
    ("jumps", "saute"),
    ("over", "par-dessus"),
    ("the", "le"),          # second "the" ↔ second "le"
    ("lazy", "paresseux"),
    ("dog", "chien"),
]

# ----------------------------
# 3) Tokenize as a pair; enable word_ids()
# ----------------------------
def split_words(s):
    # keep hyphenated "par-dessus" as one token; punctuation becomes separate tokens
    return re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9\-']+|[^\s\w]", s)

en_words_raw = split_words(en_sentence)
fr_words_raw = split_words(fr_sentence)

encoding = tokenizer(
    en_words_raw,
    fr_words_raw,
    is_split_into_words=True,   # crucial for word_ids()
    add_special_tokens=True,
    return_tensors="pt",
    return_attention_mask=True,
    return_token_type_ids=True
)

input_ids      = encoding["input_ids"]       # (1, L)
attn_mask      = encoding["attention_mask"]  # (1, L)
token_type_ids = encoding["token_type_ids"]  # (1, L) 0=EN segment, 1=FR segment

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attn_mask,
        token_type_ids=token_type_ids,
        output_attentions=True,
        output_hidden_states=True,
        return_dict=True,
    )

attentions    = outputs.attentions           # tuple len = num_layers; each (1, heads, L, L)
hidden_states = outputs.hidden_states        # tuple len = num_layers+1; 0=embeddings, 1..12 layers
num_layers    = len(attentions)

# ----------------------------
# 4) Build aligned token indices & token→word maps (robust)
# ----------------------------
seq_ids  = encoding.sequence_ids(0)  # list length L; 0 for EN, 1 for FR, None specials
word_ids = encoding.word_ids(0)      # list length L; per-segment word index, None for specials

en_tok_idx, fr_tok_idx = [], []
en_tok2word, fr_tok2word = [], []

for i, (sid, wid) in enumerate(zip(seq_ids, word_ids)):
    if sid == 0 and wid is not None:   # EN tokens (exclude specials/punct)
        en_tok_idx.append(i)
        en_tok2word.append(wid)
    elif sid == 1 and wid is not None: # FR tokens
        fr_tok_idx.append(i)
        fr_tok2word.append(wid)

# Word counts from token→word maps
num_en_words = (max(en_tok2word) + 1) if en_tok2word else 0
num_fr_words = (max(fr_tok2word) + 1) if fr_tok2word else 0

# Human-readable word labels derived from original split (filter out punctuation)
is_word = lambda t: re.match(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9\-']+$", t) is not None
en_words = [t.lower() for t in en_words_raw if is_word(t)]
fr_words = [t.lower() for t in fr_words_raw if is_word(t)]

# Safety: if tokenizer created a different count, pad/truncate labels (defensive)
if len(en_words) != num_en_words:
    en_words = (en_words + [f"en_{i}"]*num_en_words)[:num_en_words]
if len(fr_words) != num_fr_words:
    fr_words = (fr_words + [f"fr_{i}"]*num_fr_words)[:num_fr_words]

# -------- Filter placeholders from PLOTS (keep math/CSVs unchanged) --------
# Accept en_XX, fr_XX, f_XX to catch variants.
placeholder_re = re.compile(r"^(?:en|fr|f)_\d+$")

def make_plot_filter(words):
    keep = [i for i, w in enumerate(words) if not placeholder_re.fullmatch(w)]
    if not keep:  # fallback: if everything filtered (shouldn't happen), keep all
        keep = list(range(len(words)))
    old2new = {old: new for new, old in enumerate(keep)}
    words_plot = [words[i] for i in keep]
    return keep, old2new, words_plot

en_keep_idx, en_old2new, en_words_plot = make_plot_filter(en_words)
fr_keep_idx, fr_old2new, fr_words_plot = make_plot_filter(fr_words)

# ----------------------------
# 5) Token→Word aggregation helpers (robust)
# ----------------------------
def token_to_word_attention_from_joint(token_att, fr_tok_idx, en_tok_idx,
                                       fr_tok2word, en_tok2word,
                                       T_words, S_words, normalize_rows=True):
    """
    token_att: (L, L) avg over heads for one layer (self-attn over joint seq).
    Returns word-level FR→EN attention W of shape (T_words, S_words).
    """
    # Extract FR rows × EN cols block
    sub = token_att[np.ix_(fr_tok_idx, en_tok_idx)]  # (T_tokens_fr, S_tokens_en)

    # Ensure maps are aligned with sub’s axes lengths
    assert len(fr_tok2word) == sub.shape[0], "FR tok2word length mismatch with sub rows"
    assert len(en_tok2word) == sub.shape[1], "EN tok2word length mismatch with sub cols"

    W = np.zeros((T_words, S_words), dtype=float)
    for t, tw in enumerate(fr_tok2word):
        if tw is None or not (0 <= tw < T_words):  # defensive
            continue
        for s, sw in enumerate(en_tok2word):
            if sw is None or not (0 <= sw < S_words):
                continue
            W[tw, sw] += float(sub[t, s])

    if normalize_rows:
        rs = W.sum(axis=1, keepdims=True)
        rs[rs == 0.0] = 1.0
        W = W / rs
    return W

def token_to_word_embeddings_from_joint(token_emb, tok_idx, tok2word, num_words, reduce="vr"):
    """
    token_emb: (L, H) joint token embeddings for a given layer
    Returns word_emb: (num_words, H) pooled from tokens in this segment.
    """
    H = token_emb.shape[-1]
    W = np.zeros((num_words, H), dtype=np.float32)
    C = np.zeros((num_words,), dtype=np.float32)
    for i, tid in enumerate(tok_idx):
        wi = tok2word[i]
        if wi is None or not (0 <= wi < num_words):
            continue
        W[wi] += token_emb[tid]
        C[wi] += 1.0
    if reduce == "mean":
        C[C == 0.0] = 1.0
        W = W / C[:, None]
    return W

# ----------------------------
# 6) Word-level Attention per layer (FR rows × EN cols)
# ----------------------------
word_att_layers = []
for L in range(num_layers):
    att = attentions[L].mean(dim=1).squeeze(0).cpu().numpy()  # (L, L)
    W = token_to_word_attention_from_joint(
        att, fr_tok_idx, en_tok_idx, fr_tok2word, en_tok2word,
        num_fr_words, num_en_words, normalize_rows=True
    )
    word_att_layers.append(W)

# ----------------------------
# 7) Word-level Embeddings per layer (for OT)
# ----------------------------
en_word_layers, fr_word_layers = [], []
for L in range(1, num_layers + 1):  # align with attention layers 1..num_layers
    tok_emb = hidden_states[L].squeeze(0).cpu().numpy()  # (L, H)
    en_words_emb = token_to_word_embeddings_from_joint(tok_emb, en_tok_idx, en_tok2word, num_en_words, reduce="vr")
    fr_words_emb = token_to_word_embeddings_from_joint(tok_emb, fr_tok_idx, fr_tok2word, num_fr_words, reduce="vr")
    en_word_layers.append(en_words_emb)   # (EN_words, H)
    fr_word_layers.append(fr_words_emb)   # (FR_words, H)

# ----------------------------
# 8) OT cost: Euclidean + normalized Euclidean
# ----------------------------
def euclidean_distance_matrix(A, B):
    """
    A: (n, d), B: (m, d) -> D: (n, m) with ||A_i - B_j||_2
    """
    A2 = np.sum(A**2, axis=1)[:, None]
    B2 = np.sum(B**2, axis=1)[None, :]
    dist2 = A2 + B2 - 2 * A @ B.T
    dist2 = np.maximum(dist2, 0.0)
    return np.sqrt(dist2)

def euclidean_distance_matrix_normalized(A, B, eps=1e-8):
    """
    L2-normalize each vector, then compute Euclidean distance.
    For unit vectors: dist(u,v) = sqrt(2 - 2 cos_sim(u,v)).
    """
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + eps)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + eps)
    sim = A_norm @ B_norm.T
    dist = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * sim))
    return dist

# Choose OT solver
use_unregularized_emd = True   # True -> ot.emd ; False -> ot.sinkhorn
ot_reg = 0.05                   # Sinkhorn regularization (ignored if EMD)

a = np.ones((num_fr_words,), dtype=np.float64) / max(1, num_fr_words)  # FR distribution
b = np.ones((num_en_words,), dtype=np.float64) / max(1, num_en_words)  # EN distribution

ot_layers = []
ot_cost_layers = []
for L in range(num_layers):
    # Default: normalized Euclidean cost (length-invariant)
    C = euclidean_distance_matrix(fr_word_layers[L], en_word_layers[L])  # (FR_words, EN_words)
    # To use plain Euclidean instead, swap with:
    # C = euclidean_distance_matrix(fr_word_layers[L], en_word_layers[L])

    if use_unregularized_emd:
        G = ot.emd(a.astype(np.float64), b.astype(np.float64), C.astype(np.float64))
    else:
        G = ot.sinkhorn(a, b, C, reg=ot_reg)
    ot_layers.append(G)
    ot_cost_layers.append(C)

# ----------------------------
# 9) Gold pairs → indices; extract per-pair curves
# ----------------------------
def index_occurrences(words, token):
    return [i for i, w in enumerate(words) if w == token]

def ordered_word_indices(words, target_tokens):
    pos_map = {w: index_occurrences(words, w) for w in set(target_tokens)}
    cursors = {w: 0 for w in pos_map}
    idxs = []
    for w in target_tokens:
        if w in pos_map and cursors[w] < len(pos_map[w]):
            idxs.append(pos_map[w][cursors[w]])
            cursors[w] += 1
        else:
            idxs.append(None)
    return idxs

gold_en_words = [en for en, fr in gold_pairs]
gold_fr_words = [fr for en, fr in gold_pairs]
en_word_pos = ordered_word_indices(en_words, gold_en_words)
fr_word_pos = ordered_word_indices(fr_words, gold_fr_words)
pair_names = [f"{en} ↔ {fr}" for en, fr in gold_pairs]

att_pairs_vals, ot_pairs_vals = [], []
for k in range(len(gold_pairs)):
    ew, fw = en_word_pos[k], fr_word_pos[k]
    att_vals, ot_vals = [], []
    for L in range(num_layers):
        if ew is None or fw is None:
            att_vals.append(np.nan); ot_vals.append(np.nan); continue
        W_att = word_att_layers[L]; W_ot = ot_layers[L]
        att_vals.append(W_att[fw, ew] if (fw < W_att.shape[0] and ew < W_att.shape[1]) else np.nan)
        ot_vals.append(W_ot[fw,  ew] if (fw < W_ot.shape[0]  and ew < W_ot.shape[1])  else np.nan)
    att_pairs_vals.append(np.array(att_vals))
    ot_pairs_vals.append(np.array(ot_vals))

att_pairs_vals = np.stack(att_pairs_vals, axis=0)
ot_pairs_vals  = np.stack(ot_pairs_vals,  axis=0)

# ----------------------------
# 10) Outputs / plots
# ----------------------------
out_dir = "mbert_attention_ot_en_fr_fixed"
os.makedirs(out_dir, exist_ok=True)

# CSVs
with open(f"{out_dir}/gold_pairs_attention_values.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["pair"] + [f"layer_{i+1}" for i in range(num_layers)])
    for name, row in zip(pair_names, att_pairs_vals):
        w.writerow([name] + [f"{v:.6f}" if not np.isnan(v) else "" for v in row])

with open(f"{out_dir}/gold_pairs_ot_values.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["pair"] + [f"layer_{i+1}" for i in range(num_layers)])
    for name, row in zip(pair_names, ot_pairs_vals):
        w.writerow([name] + [f"{v:.6f}" if not np.isnan(v) else "" for v in row])

print(f"Saved pair CSVs to {out_dir}")

# Plot helper: matrix with BLUE circles on gold pairs (plots hide placeholders)
def plot_matrix_with_gold(M, title, filename):
    # Slice out placeholder rows/cols from the matrix for plotting
    M_plot = M[np.ix_(fr_keep_idx, en_keep_idx)]

    plt.figure(figsize=(max(6, len(en_words_plot)*0.6), max(4, len(fr_words_plot)*0.6)))
    plt.imshow(M_plot, aspect="auto")
    # plt.title(title)
    # plt.xlabel("Source (EN) words"); plt.ylabel("Target (FR) words")
    plt.xticks(range(len(en_words_plot)), en_words_plot, rotation=90)
    plt.yticks(range(len(fr_words_plot)), fr_words_plot)

    # Remap gold-pair coordinates to the filtered axes
    xs, ys = [], []
    for k in range(len(gold_pairs)):
        ew, fw = en_word_pos[k], fr_word_pos[k]
        if ew is None or fw is None:
            continue
        ew_new = en_old2new.get(ew, None)
        fw_new = fr_old2new.get(fw, None)
        if ew_new is None or fw_new is None:
            continue
        if fw_new < M_plot.shape[0] and ew_new < M_plot.shape[1]:
            xs.append(ew_new); ys.append(fw_new)

    if xs:
        plt.scatter(xs, ys, marker="o", facecolors='red', edgecolors='red', s=120, linewidth=2)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150)
    plt.close()
    print("Saved", filename)

# Per-layer attention & OT heatmaps (plots use filtered labels/axes)
for L in range(num_layers):
    plot_matrix_with_gold(
        word_att_layers[L],
        f"WORD-LEVEL Cross-Attention (Layer {L+1})",
        f"att_word_layer_{L+1:02d}.png"
    )
    metric = "OT Transport (Normalized Euclidean) " + ("[EMD]" if use_unregularized_emd else f"[Sinkhorn reg={ot_reg}]")
    plot_matrix_with_gold(
        ot_layers[L],
        f"WORD-LEVEL {metric} (Layer {L+1})",
        f"ot_word_layer_{L+1:02d}.png"
    )

# Compact pairs × layers panels (no filtering here—use full data for completeness)
plt.figure(figsize=(max(8, num_layers*0.6), max(5, len(pair_names)*0.5)))
plt.imshow(att_pairs_vals, aspect="auto")
# plt.title("Gold Pairs × Layers: Attention mass")
# plt.xlabel("Layer"); plt.ylabel("Gold pairs (EN ↔ FR)")
plt.xticks(range(num_layers), [str(i+1) for i in range(num_layers)])
plt.yticks(range(len(pair_names)), [textwrap.shorten(p, width=28, placeholder="…") for p in pair_names])
plt.colorbar(); plt.tight_layout()
plt.savefig(os.path.join(out_dir, "gold_pairs_vs_layers_attention.png"), dpi=150); plt.close()

plt.figure(figsize=(max(8, num_layers*0.6), max(5, len(pair_names)*0.5)))
plt.imshow(ot_pairs_vals, aspect="auto")
# plt.title("Gold Pairs × Layers: OT transport mass (Normalized Euclidean)")
# plt.xlabel("Layer"); plt.ylabel("Gold pairs (EN ↔ FR)")
plt.xticks(range(num_layers), [str(i+1) for i in range(num_layers)])
plt.yticks(range(len(pair_names)), [textwrap.shorten(p, width=28, placeholder="…") for p in pair_names])
plt.colorbar(); plt.tight_layout()
plt.savefig(os.path.join(out_dir, "gold_pairs_vs_layers_ot.png"), dpi=150); plt.close()

# Correlation per layer (row-normalized OT vs attention) — full matrices
def safe_corr(A, B):
    a = A.reshape(-1); b = B.reshape(-1)
    if np.allclose(a.std(), 0) or np.allclose(b.std(), 0):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

layer_corrs = []
for L in range(num_layers):
    attM = word_att_layers[L]
    otM  = ot_layers[L]
    rs = otM.sum(axis=1, keepdims=True); rs[rs == 0.0] = 1.0
    ot_norm = otM / rs
    layer_corrs.append(safe_corr(attM, ot_norm))

with open(os.path.join(out_dir, "attention_ot_layer_correlations.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["layer", "pearson_corr"])
    for i, c in enumerate(layer_corrs, 1):
        w.writerow([i, f"{c:.6f}" if not np.isnan(c) else ""])

plt.figure(figsize=(max(6, num_layers*0.6), 4))
plt.plot(range(1, num_layers+1), layer_corrs, marker="o")
# plt.title("Attention vs OT (row-normalized) Pearson correlation per layer (mBERT)")
# plt.xlabel("Layer"); plt.ylabel("Correlation")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "attention_vs_ot_layer_correlation.png"), dpi=150)
plt.close()
print("Saved per-layer correlation plot and CSV.")

# ----------------------------
# 11) Console summary
# ----------------------------
print("mBERT layers:", num_layers)
for i, (enw, frw) in enumerate(gold_pairs):
    att_row = att_pairs_vals[i]; ot_row = ot_pairs_vals[i]
    best_att_L = int(np.nanargmax(att_row)) + 1 if np.any(~np.isnan(att_row)) else None
    best_ot_L  = int(np.nanargmax(ot_row)) + 1 if np.any(~np.isnan(ot_row)) else None
    print(f"{enw:>12s} ↔ {frw:<12s} | max Attention layer={best_att_L} val={np.nanmax(att_row):.3f} | "
          f"max OT layer={best_ot_L} val={np.nanmax(ot_row):.3f}")


/u/xay7te/anaconda3/envs/gpu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Saved pair CSVs to mbert_attention_ot_en_fr_fixed
Saved att_word_layer_01.png
Saved ot_word_layer_01.png
Saved att_word_layer_02.png
Saved ot_word_layer_02.png
Saved att_word_layer_03.png
Saved ot_word_layer_03.png
Saved att_word_layer_04.png
Saved ot_word_layer_04.png
Saved att_word_layer_05.png
Saved ot_word_layer_05.png
Saved att_word_layer_06.png
Saved ot_word_layer_06.png
Saved att_word_layer_07.png
Saved ot_word_layer_07.png
Saved att_word_layer_08.png
Saved ot_word_layer_08.png
Saved att_word_layer_09.png
Saved ot_word_layer_09.png
Saved att_word_layer_10.png
Saved ot_word_layer_10.png
Saved att_word_layer_11.png
Saved ot_word_layer_11.png
Saved att_word_layer_12.png
Saved ot_word_layer_12.png
Saved per-layer correlation plot and CSV.
mBERT layers: 12
         the ↔ le           | max Attention layer=12 val=0.400 | max OT layer=1 val=0.100
       quick ↔ rapide       | max Attention layer=4 val=0.378 | max OT layer=1 val=0.100
       brown ↔ brun         | max Attention layer=4 